# Generate IDRs
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/v1/cookbook/notebooks/generate_idrs.ipynb)

Generate standalone IDRs and save them as FASTA. The prompted example uses a protein with an annotated IDR from DisProt.


## Setup
Use a standard Colab runtime, or select **Runtime → Change runtime type → GPU** and update `DEVICE` below.

Run cells in order. Installation is self-contained; no repository clone or account is needed. If Colab requests a session restart after installation, restart before running the imports. The first model load downloads weights.


In [ ]:
import sys

!"{sys.executable}" -m pip install -q "idiom[cookbook] @ git+https://github.com/rotskoff-group/idiom.git@v1"

In [1]:
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

from idiom import IDiom
from idiom.data.records import Record
from idiom.utils.notebook_helpers import save_run, write_fasta

started = time.perf_counter()

## Settings


In [2]:
MODEL_ID = "jxliu2/idiom-20M" # Pretrained IDiom model or local release directory
DEVICE = "cpu" # "cpu" or "cuda"; "auto" uses an available GPU
N = 8 # Number of standalone sequences to generate
BATCH_SIZE = 4 # Sequences per inference batch
SEED = 0 # Random seed
MAX_NEW_TOKENS = 128 # Generation limit, including STOP; not a fixed IDR length
OUT_DIR = Path("generation_outputs") / time.strftime("%Y%m%d-%H%M%S") # New timestamped folder per run
PROMPT_FASTA = None # Default: annotated DisProt proteins; otherwise set a FASTA path

## Generate


In [3]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
model = IDiom.from_pretrained(MODEL_ID, device=DEVICE)
sequences = model.generate_unprompted(n=N, batch_size=BATCH_SIZE, seed=SEED, max_new_tokens=MAX_NEW_TOKENS)

## Inspect and save


In [4]:
table = pd.DataFrame({"sequence": sequences, "length": [len(s) for s in sequences]})
with pd.option_context("display.max_colwidth", 80):
    display(table.head(3))
table.to_csv(OUT_DIR / "sequences.csv", index=False)
_ = write_fasta(
    [Record(f"generated_{i}", s, 0, len(s)) for i, s in enumerate(sequences) if s],
    OUT_DIR / "generated.fasta",
)

,sequence,length
0,DKKEWGNEEEHEDSGGRPHAPPQASSIQVL,30
1,MATANGAQKNSPRNFIPWLVFTLGFSGGYLAYRAFYLKQQVLLHRGGSAPIPAIRGHRAATMSLPDAPGTPTGSDT...,127
2,DLLQREIIEGTGTQTIDSRALKECGNNVNQETEDGYIPGNEKSANIYDIVQMVKESNTNVVVR,63


## Variant: generate between protein flanks
Use the first record in an annotated FASTA (`>protein_IDR_3-18`, 1-based inclusive), or use the first annotated IDR in the bundled DisProt example data. The API takes 0-based, end-exclusive coordinates and returns only the generated IDR. The export restores the unchanged flanks.


In [5]:
from idiom.utils.notebook_helpers import check_context, example_file, load_inputs

prompt_path = PROMPT_FASTA or example_file("disprot/disprot_len1020_idrs.fasta", Path("example_inputs"))
prompts, audit = load_inputs(prompt_path, "annotated", limit=1)
audit.to_csv(OUT_DIR / "prompt_input_audit.csv", index=False)
if not prompts:
    raise ValueError("No valid annotated protein; review prompt_input_audit.csv.")
prompt = prompts[0]
check_context([prompt], model.model.cfg.max_seq_len, include_flanks=True)
display(audit.loc[audit.status == "accepted"])
prompted = model.generate_prompted(
    prompt.full_seq,
    prompt.idr_start,
    prompt.idr_end,
    n=2,
    batch_size=BATCH_SIZE,
    seed=SEED,
    max_new_tokens=MAX_NEW_TOKENS,
)
prompted_table = pd.DataFrame(
    {
        "generated_id": [f"redesign_{i}" for i in range(len(prompted))],
        "original_idr": prompt.full_seq[prompt.idr_start : prompt.idr_end],
        "generated_idr": prompted,
        "length": [len(s) for s in prompted],
    }
)
prompted_table.to_csv(OUT_DIR / "prompted_sequences.csv", index=False)
with pd.option_context("display.max_colwidth", 80):
    display(prompted_table.head(3))
write_fasta(
    [Record(f"redesign_{i}", s, 0, len(s)) for i, s in enumerate(prompted) if s],
    OUT_DIR / "prompted_idrs.fasta",
)
redesigned = [
    Record(
        f"redesign_{i}",
        prompt.full_seq[: prompt.idr_start] + s + prompt.full_seq[prompt.idr_end :],
        prompt.idr_start,
        prompt.idr_start + len(s),
    )
    for i, s in enumerate(prompted)
    if s
]
_ = write_fasta(redesigned, OUT_DIR / "redesigned_proteins.fasta")

,record_id,accession,header,idr_start_1based,idr_end_1based,status
0,record_0,P03265,P03265_IDR_294-334,294,334,accepted


,generated_id,original_idr,generated_idr,length
0,redesign_0,EHVIEMDVTSENGQRALKEQSSKAKIVKNRWGRNVVQISNT,DKKRWGNEAEHEASGGRPHAPPQASSIQVL,30
1,redesign_1,EHVIEMDVTSENGQRALKEQSSKAKIVKNRWGRNVVQISNT,EAMANGAQKNSPRVFHPWEVFTLGFSGGYLKERIFYLFQQVLLHRGGSAPIPSIRGHRGATMSLPDYPGTPSYSDT...,128


## Results
`sequences.csv` and `generated.fasta` contain standalone samples. `prompted_sequences.csv`, `prompted_idrs.fasta`, and `redesigned_proteins.fasta` contain the prompted samples and reconstructed proteins. Empty generations stay in the tables and are omitted from FASTA exports.

`run.json` records the settings and package versions. Open the output folder in Colab’s **Files** pane to download results. Download them before the runtime ends, or copy them to mounted Drive. Saved notebook previews do not include the exported files.


Saved previews show a CPU example run. Run the cells to create the exported files.

In [6]:
save_run(
    OUT_DIR,
    dict(
        model=MODEL_ID,
        device=DEVICE,
        seed=SEED,
        n=N,
        batch_size=BATCH_SIZE,
        max_new_tokens=MAX_NEW_TOKENS,
        prompt_fasta=str(prompt_path),
    ),
    elapsed=time.perf_counter() - started,
)
print("Results folder:", OUT_DIR)

Results folder: generation_outputs/20260921-134131
